# Case 01: Olist 巴西電商完整分析

## 📊 資料集介紹

**Olist Brazilian E-Commerce Dataset**
- 100K 筆訂單（2016-2018）
- 完整電商流程：訂單 → 客戶 → 商品 → 評論 → 物流
- 巴西最大的電商平台之一

## 🎯 學習目標

1. **多表合併** - 使用 `merge` 整合 9 個資料表
2. **時間序列分析** - 訂單趨勢、成長率
3. **客戶分析** - RFM 初探、複購率
4. **商品分析** - 熱銷排名、類別表現
5. **評論分析** - 滿意度、文字分析

---

In [ ]:
# 環境設定
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# 資料路徑
DATA_DIR = Path('/mnt/data/datasets/ecommerce/kaggle/olist')

print("✅ 環境設定完成！")
print(f"📁 資料目錄: {DATA_DIR}")

---

## 📥 Step 1: 載入所有資料表

Olist 資料集包含 9 個表，我們先全部載入並快速檢視：

In [ ]:
# 載入所有資料表
orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
order_items = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA_DIR / 'olist_order_reviews_dataset.csv')
geolocation = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')
category_translation = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

print("✅ 所有資料表載入完成！\n")

# 快速檢視各表大小
tables = {
    'orders': orders,
    'order_items': order_items,
    'customers': customers,
    'products': products,
    'sellers': sellers,
    'payments': payments,
    'reviews': reviews,
    'geolocation': geolocation,
    'category_translation': category_translation
}

summary = pd.DataFrame({
    '資料表': tables.keys(),
    '筆數': [len(df) for df in tables.values()],
    '欄位數': [len(df.columns) for df in tables.values()]
})

print(summary.to_string(index=False))

---

## 🔍 Step 2: 快速探索 - 訂單表

先了解最重要的訂單表：

In [ ]:
# 查看訂單表結構
print("📊 訂單表資訊：")
print(orders.info())
print("\n前 5 筆資料：")
orders.head()

In [ ]:
# 訂單狀態分佈
print("📦 訂單狀態分佈：\n")
print(orders['order_status'].value_counts())

# 視覺化
plt.figure(figsize=(10, 6))
orders['order_status'].value_counts().plot(kind='barh', color='steelblue')
plt.title('訂單狀態分佈', fontsize=14, fontweight='bold')
plt.xlabel('訂單數')
plt.ylabel('狀態')
plt.tight_layout()
plt.show()

---

## 🔗 Step 3: 多表合併（進階 Merge）

**這是關鍵技能！** 將分散的資料整合成完整的寬表：

In [ ]:
# 先處理日期欄位
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print("✅ 日期欄位轉換完成")

In [ ]:
# 步驟 1: 訂單 + 客戶
df = orders.merge(
    customers, 
    on='customer_id', 
    how='left'
)

print(f"✅ 訂單 + 客戶: {len(df):,} 筆")

# 步驟 2: + 訂單明細
df = df.merge(
    order_items, 
    on='order_id', 
    how='left'
)

print(f"✅ + 訂單明細: {len(df):,} 筆")

# 步驟 3: + 商品資訊
df = df.merge(
    products, 
    on='product_id', 
    how='left'
)

print(f"✅ + 商品資訊: {len(df):,} 筆")

# 步驟 4: + 類別翻譯
df = df.merge(
    category_translation, 
    on='product_category_name', 
    how='left'
)

print(f"✅ + 類別翻譯: {len(df):,} 筆")

# 步驟 5: + 付款資訊（注意：一筆訂單可能有多筆付款）
payment_summary = payments.groupby('order_id').agg({
    'payment_value': 'sum',
    'payment_type': lambda x: ', '.join(x.unique())
}).reset_index()

payment_summary.columns = ['order_id', 'total_payment', 'payment_methods']

df = df.merge(
    payment_summary, 
    on='order_id', 
    how='left'
)

print(f"✅ + 付款資訊: {len(df):,} 筆")

# 步驟 6: + 評論資訊
review_summary = reviews[['order_id', 'review_score']]

df = df.merge(
    review_summary, 
    on='order_id', 
    how='left'
)

print(f"✅ + 評論資訊: {len(df):,} 筆")

print("\n🎉 完整寬表建立完成！")
print(f"📊 最終維度: {df.shape}")

In [ ]:
# 查看合併後的資料
print("合併後的欄位：\n")
print(df.columns.tolist())
print("\n前 3 筆資料：")
df.head(3)

---

## 📈 Step 4: 時間序列分析

分析訂單趨勢、成長率：

In [ ]:
# 只保留已完成的訂單
df_delivered = df[df['order_status'] == 'delivered'].copy()

# 新增年月欄位
df_delivered['year_month'] = df_delivered['order_purchase_timestamp'].dt.to_period('M')

# 每月訂單數
monthly_orders = df_delivered.groupby('year_month').agg({
    'order_id': 'nunique',
    'price': 'sum'
}).reset_index()

monthly_orders.columns = ['年月', '訂單數', '總營收']
monthly_orders['年月'] = monthly_orders['年月'].astype(str)

print(monthly_orders)

In [ ]:
# 視覺化趨勢
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 訂單數趨勢
axes[0].plot(monthly_orders['年月'], monthly_orders['訂單數'], 
             marker='o', linewidth=2, color='steelblue')
axes[0].set_title('每月訂單數趨勢', fontsize=14, fontweight='bold')
axes[0].set_xlabel('年月')
axes[0].set_ylabel('訂單數')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# 營收趨勢
axes[1].plot(monthly_orders['年月'], monthly_orders['總營收'], 
             marker='s', linewidth=2, color='green')
axes[1].set_title('每月營收趨勢', fontsize=14, fontweight='bold')
axes[1].set_xlabel('年月')
axes[1].set_ylabel('營收（BRL）')
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---

## 🛍️ Step 5: 商品分析

找出熱銷產品類別：

In [ ]:
# Top 20 產品類別
category_sales = df_delivered.groupby('product_category_name_english').agg({
    'order_id': 'count',
    'price': 'sum'
}).reset_index()

category_sales.columns = ['類別', '訂單數', '總營收']
category_sales = category_sales.sort_values('訂單數', ascending=False).head(20)

print("📦 Top 20 熱銷類別：\n")
print(category_sales.to_string(index=False))

In [ ]:
# 視覺化
plt.figure(figsize=(12, 8))
plt.barh(category_sales['類別'], category_sales['訂單數'], color='coral')
plt.xlabel('訂單數')
plt.title('Top 20 熱銷產品類別', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---

## ⭐ Step 6: 評論分析

分析客戶滿意度：

In [ ]:
# 評分分佈
print("⭐ 評分分佈：\n")
print(df_delivered['review_score'].value_counts().sort_index())

# 視覺化
plt.figure(figsize=(10, 6))
df_delivered['review_score'].value_counts().sort_index().plot(
    kind='bar', color='gold', edgecolor='black'
)
plt.title('客戶評分分佈', fontsize=14, fontweight='bold')
plt.xlabel('評分')
plt.ylabel('數量')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 各類別平均評分
category_rating = df_delivered.groupby('product_category_name_english')['review_score'].agg([
    'mean', 'count'
]).reset_index()

category_rating.columns = ['類別', '平均評分', '評論數']
category_rating = category_rating[category_rating['評論數'] >= 100]  # 只看評論數 >= 100 的類別
category_rating = category_rating.sort_values('平均評分', ascending=False).head(15)

print("⭐ 評分最高的產品類別（評論數 >= 100）：\n")
print(category_rating.to_string(index=False))

---

## 💡 Step 7: 客戶分析入門

計算複購率、客單價：

In [ ]:
# 每位客戶的訂單統計
customer_stats = df_delivered.groupby('customer_unique_id').agg({
    'order_id': 'nunique',  # 訂單數
    'total_payment': 'sum',  # 總消費
    'order_purchase_timestamp': ['min', 'max']  # 首購、末購日期
}).reset_index()

customer_stats.columns = ['客戶ID', '訂單數', '總消費', '首購日期', '末購日期']

# 計算客單價
customer_stats['平均客單價'] = customer_stats['總消費'] / customer_stats['訂單數']

print("👥 客戶統計摘要：\n")
print(customer_stats.describe())

In [ ]:
# 複購率分析
repeat_rate = (customer_stats['訂單數'] > 1).mean() * 100

print(f"\n📊 複購率: {repeat_rate:.2f}%")
print(f"📊 平均訂單數: {customer_stats['訂單數'].mean():.2f}")
print(f"📊 平均總消費: R$ {customer_stats['總消費'].mean():.2f}")
print(f"📊 平均客單價: R$ {customer_stats['平均客單價'].mean():.2f}")

In [ ]:
# 訂單數分佈
plt.figure(figsize=(10, 6))
customer_stats['訂單數'].value_counts().sort_index().head(10).plot(
    kind='bar', color='teal'
)
plt.title('客戶訂單數分佈（Top 10）', fontsize=14, fontweight='bold')
plt.xlabel('訂單數')
plt.ylabel('客戶數')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---

## 🎯 總結與下一步

### ✅ 完成的學習目標：

1. ✅ **多表合併** - 使用 `merge` 整合 9 個資料表成完整寬表
2. ✅ **時間序列** - 分析訂單趨勢、營收成長
3. ✅ **商品分析** - 找出熱銷類別
4. ✅ **評論分析** - 了解客戶滿意度
5. ✅ **客戶分析** - 計算複購率、客單價

### 🚀 進階練習建議：

1. **RFM 分析** - 客戶分群（下一個案例）
2. **Cohort 分析** - 留存率分析
3. **物流分析** - 配送時間優化
4. **地理分析** - 使用 geolocation 資料
5. **預測模型** - 預測評分、銷售額

---

## 💾 儲存清洗後的資料

供後續案例使用：

In [ ]:
# 儲存完整寬表
output_path = Path('/mnt/data/datasets/ecommerce/processed/olist_complete.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(output_path, index=False)

print(f"✅ 資料已儲存至: {output_path}")
print(f"💾 檔案大小: {output_path.stat().st_size / (1024**2):.2f} MB")